[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Why sqlite3


## What you will be able to do

Create a SQLite database with Python's standard library, load a CSV file into a table once, and
answer questions about the data with SQL instead of reading the file again for every question. Open
the database from a new connection, recognize it as a single file you can copy, and read the errors
SQLite gives when a path points at the wrong file, or at no file at all.


## The idea

### The problem

Four weather stations, Bergen, Oslo, Svalbard and Tromso, send a temperature every hour, and a year
of it sits in `readings.csv`: 35,040 lines, each holding a station, an hour and a reading in degrees
Celsius. On one day in March the station on Svalbard sent nothing, so its cells for that day are
empty.

Every question about that year starts the same way. What was the coldest hour on Svalbard? A script
opens the file, splits all 35,040 lines, turns every reading from text into a number, skips the empty
ones, and keeps the smallest. What was Oslo's mean temperature in July? Another script opens the same
file, splits the same lines and converts the same numbers, to keep 744 of them. The whole file is
read for every question, even a question about one hour, and every script carries its own copy of
the parsing, with its own chance of treating an empty cell as zero.

Sharing the file makes it worse. A program appending tomorrow's readings while another reads can
leave the reader with half a line. A new column breaks every script that picked values out by
position. The data is stored as text that has to be understood again whenever anyone looks at it.

### What a database is

> A **database** keeps data in a form a program can ask questions of, without reading all of it and
> without parsing it again. A **relational database** keeps it in **tables**, each made of **rows**
> that share the same named **columns**, and answers questions written in **SQL**, a language for
> saying which rows you want rather than how to find them. **SQLite** is a relational database that
> runs inside your program instead of as a separate server, and keeps a whole database, every table
> in it, in one ordinary file. Python's **sqlite3** module, part of the standard library, connects a
> program to it: `sqlite3.connect(path)` opens that file, creating it when it does not exist, and
> returns a **connection**, whose `execute` method runs one SQL statement.

### Why it works that way

- **Parse once, then ask.** Loading the CSV into a table turns every reading into a number a single
  time. After that, a query reads stored numbers, never text that has to be split and converted.
- **You say what you want, and SQLite decides how.** A query names the rows and the summary it
  wants, and SQLite works out how to read them. An index lets it find one station's rows without
  reading the others, which the **Indexes and Query Plans** notebook measures.
- **One file, and no server.** There is nothing to install and nothing to start. Copy the file and
  you have copied the database, provided nothing is writing to it at that moment, a condition the
  **Backup and Copying** notebook makes exact.
- **A change happens completely or not at all.** Writes are grouped into a transaction, and a
  program that dies halfway through leaves the file as it was before the transaction began. The
  **Transactions** notebook shows what that asks of your code.
- **Values travel separately from the SQL.** A reading goes into a statement through a `?`
  placeholder, never by pasting its text into the SQL, which the **Parameters** notebook shows is how
  injection happens.
- **Missing is a value of its own.** An empty reading is stored as `NULL`, which SQL's summaries
  skip. Stored as an empty string instead, it would be counted, and a mean would treat it as zero.

### Where this shows up

SQLite is the most widely deployed database engine there is. Firefox and Chrome keep their browsing
history in SQLite files, Android and iOS apps store their local data in it, and IPython keeps its
command history in one. On a server it holds a small application's data, where a separate database
server would be one more thing to run and look after. In this library, the **Files, Paths and
Formats** guide read and wrote CSV and JSON files, which a database replaces once the same questions
keep being asked. The **SQLAlchemy, Deep Dive**, **SQLModel, Deep Dive** and **Peewee, Deep Dive**
guides all run on SQLite, and **DuckDB, Deep Dive** takes the other side: SQL over CSV and Parquet
files, where they sit.

### The vocabulary of sqlite3

This guide works from the file up: what a database holds and how a statement reads it, how Python
talks to it and moves values in and out, and how the database keeps its data correct, fast and safe.
The tables below are the map, and their last column names the notebook that covers each term in
depth.

The database and its SQL:

| Term | What it is | Example | Covered in depth in |
|---|---|---|---|
| Database file | one file holding every table, beginning with the header `SQLite format 3` | `stations.db` | this notebook, and **Backup and Copying** |
| Table | a named set of rows that share the same columns | `readings` | **Tables and Queries** |
| Row and column | one record, and one named value in every record | `('Svalbard', '2025-03-02T06:00', None)` | **Tables and Queries** |
| SQL statement | one instruction to the database | `CREATE TABLE readings (...)` | **Tables and Queries** |
| Query | a statement that reads rows and hands them back | `SELECT station FROM readings` | this notebook, and **Tables and Queries** |
| `NULL` | the value for no value, which summaries such as `AVG` skip | a missing reading | this notebook, and **Type Affinity** |
| Type affinity | the type a column prefers, which SQLite applies to what you store | `REAL` turning `'4.2'` into `4.2` | **Type Affinity** |
| Constraint | a rule the database refuses to break | `NOT NULL`, `UNIQUE`, a foreign key | **Constraints** |
| Schema | the tables and columns a database holds, and how they change | `ALTER TABLE` | **Changing a Schema** |
| Index | a sorted copy of some columns that lets a query skip most rows | `CREATE INDEX` | **Indexes and Query Plans** |
| Full-text search | a table built for finding words in text, ranked by relevance | FTS5 | **Full-Text Search** |

Talking to it from Python:

| Term | What it is | Example | Covered in depth in |
|---|---|---|---|
| Connection | an open database file, and the object every statement goes through | `sqlite3.connect("stations.db")` | **Connections and Cursors** |
| Cursor | the object that runs a statement and walks the rows it returns | `cursor.fetchone()` | **Connections and Cursors** |
| Placeholder | a `?` in the SQL, filled by a value passed alongside it | `WHERE station = ?` | this notebook, and **Parameters** |
| Row factory | what a row comes back as: a tuple, or something you can index by column name | `sqlite3.Row` | **Row Factories** |
| Adapter and converter | the functions that turn a Python value into a stored one, and back | a `datetime` | **Adapters and Converters** |
| `executemany` | one statement, run for every item in a sequence of rows | loading a year of readings | this notebook, and **executemany** |

Keeping the data correct, fast and safe:

| Term | What it is | Example | Covered in depth in |
|---|---|---|---|
| Transaction | a group of changes saved together, or not at all | the rows of one load | **Transactions** |
| Commit and rollback | saving a transaction's changes, or throwing them away | `conn.commit()` | **Transactions** |
| autocommit | the connection setting that decides when a transaction begins | `autocommit=False` | **autocommit and isolation_level** |
| Lock and WAL | how SQLite lets readers work while one writer writes | `database is locked` | **Concurrency and WAL** |
| Backup | a consistent copy, taken while the database is in use | `conn.backup(target)` | **Backup and Copying** |

### What this notebook covers

- A database, made by connecting to a file that does not exist yet
- A table, and a year of readings loaded into it once
- Questions about that year, answered with SQL
- The same database opened again, by a new connection and as a copied file
- A new day's readings added to the database, and the same questions asked of it
- Four errors: a folder that does not exist, a CSV opened as a database, a misspelled file name that
  makes a new empty database, and empty readings that a mean counts as zero degrees

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3
import tempfile
from pathlib import Path

path = Path(tempfile.mkdtemp()) / "stations.db"

conn = sqlite3.connect(path)
conn.execute("CREATE TABLE readings (station TEXT, hour TEXT, celsius REAL)")
conn.executemany("INSERT INTO readings VALUES (?, ?, ?)", [
    ("Bergen", "2025-01-14T06:00", 4.2), ("Bergen", "2025-01-14T07:00", 3.9),
    ("Svalbard", "2025-01-14T06:00", -18.5), ("Svalbard", "2025-01-14T07:00", -19.1),
])
conn.commit()
conn.close()

query = "SELECT station, MIN(celsius) FROM readings GROUP BY station ORDER BY station"
conn = sqlite3.connect(path)          # a new connection, as tomorrow's program would open
for station, coldest in conn.execute(query):
    print(station, coldest)
conn.close()
print(path.name, "is one file of", path.stat().st_size, "bytes")
```

```
Bergen 3.9
Svalbard -19.1
stations.db is one file of 8192 bytes
```

Four readings, written to a table in a file and committed, then read back by a connection that had
nothing in memory, the way a program run tomorrow would read them. SQLite, not a Python loop, found
the coldest reading at each station. And the database, with its table and its rows, is one file.


## Setup

Six imports, a folder to work in, and a year of readings to load.

- `sqlite3` opens the database and runs every statement
- `csv` writes the year of readings to a CSV file, and reads it back to load it
- `math` shapes the readings into seasons and days, so that every run writes the same file
- `datetime` and `timedelta` count out the hours of the year
- `Path` names the files, and reads the first bytes of the database file
- `shutil` copies the database file, and removes the scratch folder at the end

Setup writes `scratch/readings.csv`: every hour of 2025 at the four stations, with Svalbard's
readings empty for one day in March. The readings follow a formula rather than random numbers, so the
file, and every number this notebook prints, is the same on every computer.


In [1]:
import csv
import math
import shutil
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
READINGS = SCRATCH / "readings.csv"
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year

with open(READINGS, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["station", "hour", "celsius"])
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))     # coldest in the middle of January
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)          # coldest a little before dawn
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = ""                                             # the day the station sent nothing
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10              # the same small variation on every run
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            writer.writerow([station, hour.strftime("%Y-%m-%dT%H:%M"), celsius])

print("wrote", READINGS, "with", len(READINGS.read_text(encoding="utf-8").splitlines()) - 1, "readings")


wrote scratch/readings.csv with 35040 readings


## Worked examples

### A database in one file

`sqlite3.connect` takes the path of a database file. When the file does not exist, SQLite creates it,
so the first connection to a new path is also how a database is made. The connection is the object
every statement goes through:


In [2]:
DATABASE = SCRATCH / "stations.db"
conn = sqlite3.connect(DATABASE)

print("connected:", type(conn).__name__)
print("files in scratch:", sorted(path.name for path in SCRATCH.iterdir()))
print(DATABASE.name, "holds", DATABASE.stat().st_size, "bytes so far")


connected: Connection
files in scratch: ['readings.csv', 'stations.db']
stations.db holds 0 bytes so far


The file exists as soon as the connection does, and it is empty: SQLite writes nothing until there
is something to store. The **Connections and Cursors** notebook looks at what a connection holds,
and why closing one matters.

### A table, and a year of readings loaded into it once

A table has to exist before rows can go into it. `CREATE TABLE` names its columns and the kind of
value each holds: `TEXT` for the station and the hour, and `REAL`, a floating point number, for the
reading. `NOT NULL` makes SQLite refuse a row with no station or no hour, while a reading is allowed
to be missing. `PRAGMA table_info` asks SQLite to describe the table it now holds:


In [3]:
conn.execute("""
    CREATE TABLE readings (
        station TEXT NOT NULL,
        hour    TEXT NOT NULL,
        celsius REAL
    )
""")

print([(column[1], column[2]) for column in conn.execute("PRAGMA table_info(readings)")])


[('station', 'TEXT'), ('hour', 'TEXT'), ('celsius', 'REAL')]


Now the CSV, read once. `csv.DictReader` hands back every line as a dictionary of strings, and
`reading` turns one of those into the values a row of the table holds: the reading becomes a `float`,
and an empty reading becomes `None`, which is how Python passes SQL's `NULL`. `executemany` runs the
one `INSERT` statement for every row the reader produces, and each `?` in it is a placeholder that
the row's values fill:


In [4]:
def reading(row):
    """One line of the CSV as the values of one row of readings. An empty reading becomes None."""
    celsius = float(row["celsius"]) if row["celsius"] else None
    return row["station"], row["hour"], celsius


with open(READINGS, newline="", encoding="utf-8") as file:
    conn.executemany("INSERT INTO readings (station, hour, celsius) VALUES (?, ?, ?)",
                     map(reading, csv.DictReader(file)))
conn.commit()

print("rows in the table:", conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0])


rows in the table: 35040


`commit` makes the load permanent. Until it runs, the new rows exist only for this connection, and a
program that stopped before reaching it would leave the table as empty as it was. The
**Transactions** notebook shows exactly when rows are saved, and the **executemany** notebook what
running one statement for many rows saves.

### Questions, answered with SQL

A query begins with `SELECT`, names the columns it wants back, and picks rows with `WHERE`. Here is
the coldest hour on Svalbard. `ORDER BY celsius, hour` sorts the rows by their reading, and by their
hour when two readings are equal, and `LIMIT 1` keeps the first. SQLite sorts `NULL` before every
number, so without `celsius IS NOT NULL` the coldest hour is an hour with no reading at all, which
the second line shows. The station goes in through a placeholder, with its value in the tuple passed
after the SQL, and the condition is the one piece of this SQL that the cell writes itself:


In [5]:
COLDEST = """
    SELECT hour, celsius FROM readings
    WHERE station = ? {}
    ORDER BY celsius, hour
    LIMIT 1
"""

print("coldest hour on Svalbard:  ", conn.execute(COLDEST.format("AND celsius IS NOT NULL"), ("Svalbard",)).fetchone())
print("without the IS NOT NULL:   ", conn.execute(COLDEST.format(""), ("Svalbard",)).fetchone())


coldest hour on Svalbard:   ('2025-01-12T03:00', -17.3)
without the IS NOT NULL:    ('2025-03-02T00:00', None)


`GROUP BY` puts the rows of one station together, and an aggregate function turns every group into
one value: `MIN` and `MAX` the extremes, `AVG` the mean, and `COUNT` the number of rows. `COUNT(*)`
counts rows, and `COUNT(celsius)` counts only the rows that hold a reading. Every station's year, in
one statement:


In [6]:
yearly = conn.execute("""
    SELECT station, MIN(celsius), MAX(celsius), AVG(celsius), COUNT(*), COUNT(celsius)
    FROM readings
    GROUP BY station
    ORDER BY station
""")

print(f"{'station':<10}{'coldest':>8}{'warmest':>8}{'mean':>7}{'hours':>7}{'readings':>9}")
for station, coldest, warmest, mean, hours, readings in yearly:
    print(f"{station:<10}{coldest:>8}{warmest:>8}{mean:>7.1f}{hours:>7}{readings:>9}")


station    coldest warmest   mean  hours readings
Bergen        -4.8    20.8    8.0   8760     8760
Oslo          -6.3    19.3    6.5   8760     8760
Svalbard     -17.3     8.3   -4.5   8760     8736
Tromso        -9.2    16.2    3.5   8760     8760


Svalbard has as many hours as the other stations and 24 fewer readings, the day it sent nothing, and
its mean was taken over the readings that exist. A query can group by something it works out, too.
`substr(hour, 1, 7)` keeps the year and month of an hour, so grouping by it gives Oslo's mean for
each month:


In [7]:
monthly = conn.execute("""
    SELECT substr(hour, 1, 7) AS month, AVG(celsius)
    FROM readings
    WHERE station = ?
    GROUP BY month
    ORDER BY month
""", ("Oslo",))

for month, mean in monthly:
    print(month, f"{mean:5.1f}")


2025-01  -2.4
2025-02  -1.4
2025-03   1.7
2025-04   6.0
2025-05  10.6
2025-06  14.0
2025-07  15.4
2025-08  14.4
2025-09  11.2
2025-10   6.8
2025-11   2.3
2025-12  -1.1


None of these questions opened `readings.csv`, split a line or converted a string. They read numbers
the table already held, and a new question is a new statement rather than a new script. The **Tables
and Queries** notebook takes `SELECT`, `WHERE` and `GROUP BY` further, to queries across more than
one table.

### The same database, from a new connection

Closing the connection leaves the notebook holding nothing of the database in memory. A new
connection opens the file the way a program run tomorrow would, and the table is there, with its
readings stored as numbers. `typeof` reports how SQLite stored a value:


In [8]:
conn.close()

tomorrow = sqlite3.connect(DATABASE)
print("rows:", tomorrow.execute("SELECT COUNT(*) FROM readings").fetchone()[0])
print("stored as:", tomorrow.execute(
    "SELECT typeof(station), typeof(hour), typeof(celsius) FROM readings WHERE celsius IS NOT NULL LIMIT 1"
).fetchone())
print("a missing reading stored as:", tomorrow.execute(
    "SELECT typeof(celsius) FROM readings WHERE celsius IS NULL LIMIT 1"
).fetchone())
tomorrow.close()


rows: 35040
stored as: ('text', 'text', 'real')
a missing reading stored as: ('null',)


The readings are `real`, numbers, and the missing ones are `null`. A CSV holds only text, so every
program that reads one has to know which of its columns are numbers. The table knows. The **Type
Affinity** notebook shows where that knowledge ends, because SQLite, unlike most databases, will also
store text in a `REAL` column.

### One file you can copy

Everything the database holds, the table's definition and every row, is in that one file. Its first
bytes say what it is, where the CSV's first bytes are simply its header line:


In [9]:
print(DATABASE.name, DATABASE.read_bytes()[:16])
print(READINGS.name, READINGS.read_bytes()[:16])


stations.db b'SQLite format 3\x00'
readings.csv b'station,hour,cel'


A copy of the file is a copy of the database, and it opens exactly as the original does:


In [10]:
copy = SCRATCH / "stations-copy.db"
shutil.copy(DATABASE, copy)

other = sqlite3.connect(copy)
print("rows in the copy:", other.execute("SELECT COUNT(*) FROM readings").fetchone()[0])
other.close()


rows in the copy: 35040


The copy was safe because no connection had the file open. Copying a file while a program writes to
it can catch half of a change, and the **Backup and Copying** notebook shows that happen, then shows
the backup that cannot.

### A new day, and the same questions

The pieces of this notebook, in the shape of a job that runs every day. A new day's readings arrive
as a CSV, `load` adds them to the database, and `ask` puts a question to the whole file, which
includes the new day at once. `load` and `ask` open a connection of their own and close it, as two
separate programs would:


In [11]:
def load(csv_path, database):
    """Add the readings in a CSV file to the database, and return how many rows were added."""
    conn = sqlite3.connect(database)
    with open(csv_path, newline="", encoding="utf-8") as file:
        cursor = conn.executemany("INSERT INTO readings (station, hour, celsius) VALUES (?, ?, ?)",
                                  map(reading, csv.DictReader(file)))
    conn.commit()
    added = cursor.rowcount
    conn.close()
    return added


def ask(database, sql, parameters=()):
    """Run one query against the database, and return all of its rows."""
    conn = sqlite3.connect(database)
    rows = conn.execute(sql, parameters).fetchall()
    conn.close()
    return rows


The first day of 2026 arrives, 24 hours at the four stations, and goes into the same table:


In [12]:
new_day = SCRATCH / "2026-01-01.csv"
with open(new_day, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["station", "hour", "celsius"])
    for hour in range(24):
        for station, celsius in [("Bergen", 1.5), ("Oslo", -6.0), ("Svalbard", -21.0), ("Tromso", -9.5)]:
            writer.writerow([station, f"2026-01-01T{hour:02d}:00", round(celsius - hour % 5 / 10, 1)])

print("added:", load(new_day, DATABASE))
print("coldest hour since the new year:", ask(DATABASE, """
    SELECT station, hour, celsius FROM readings
    WHERE hour >= ? AND celsius IS NOT NULL
    ORDER BY celsius, hour
    LIMIT 1
""", ("2026-01-01",)))
print("hours at each station:", ask(DATABASE, "SELECT station, COUNT(*) FROM readings GROUP BY station ORDER BY station"))


added: 96
coldest hour since the new year: [('Svalbard', '2026-01-01T04:00', -21.4)]
hours at each station: [('Bergen', 8784), ('Oslo', 8784), ('Svalbard', 8784), ('Tromso', 8784)]


The new day's CSV was read once, by `load`. Every question after that, about one day or the whole
file, was answered from the database, and `WHERE hour >= ?` worked on text because an hour written as
year, month, day and time sorts in the order of time.

### Where each part came from

| In the job | What it relies on | The section that showed it |
|---|---|---|
| `stations.db`, opened by `load` and by `ask` | a database that is one file, made the first time something connects to it | A database in one file |
| `executemany` with `?` placeholders, then `commit` | rows loaded once, and saved | A table, and a year of readings loaded into it once |
| `reading`, turning an empty cell into `None` | a missing reading stored as `NULL` | A table, and a year of readings loaded into it once |
| `ORDER BY celsius, hour` with `LIMIT 1`, and `GROUP BY station` | questions answered by SQL rather than by a loop | Questions, answered with SQL |
| a connection opened and closed inside every function | the database read from its file, with nothing kept in memory | The same database, from a new connection |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/01-why-sqlite3-solutions.ipynb).

**1.** Connect to `scratch/stations.db` and print the warmest hour of the year at Bergen, with its
reading.


In [13]:
# your code here


**2.** Print Tromso's mean temperature for July 2025, rounded to one decimal place.


In [14]:
# your code here


**3.** Print, for every station, how many of its hours have no reading.


In [15]:
# your code here


**4.** Print, for every station, how many of its readings in 2025 were below freezing.


In [16]:
# your code here


**5.** Add a table called `stations` to the database, with a `name` and a `latitude`, holding Bergen
at 60.39, Oslo at 59.91, Svalbard at 78.22 and Tromso at 69.65. Commit, then print its rows in order
of name from a new connection.


In [17]:
# your code here


**6.** Copy `scratch/stations.db` to `scratch/task-copy.db`. Print whether the copy begins with
SQLite's header, and whether its `readings` table holds as many rows as the original's.


In [18]:
# your code here


## Common errors

### sqlite3.OperationalError: unable to open database file


In [19]:
archive = SCRATCH / "archive" / "2025" / "stations.db"

conn = sqlite3.connect(archive)


OperationalError: unable to open database file

SQLite creates a database file that does not exist, but not the folders it goes in. `scratch/archive`
does not exist, so there was nowhere to put `stations.db`, and the message does not say which part of
the path was missing. Make the folders first:


In [20]:
archive.parent.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(archive)
print("opened:", archive.exists())
conn.close()


opened: True


### sqlite3.DatabaseError: file is not a database


In [21]:
wrong = sqlite3.connect(READINGS)

wrong.execute("SELECT COUNT(*) FROM readings")


DatabaseError: file is not a database

`connect` raised nothing, because SQLite does not read a file until the first statement needs it.
That statement found `station,hour,cel` where `SQLite format 3` should have been, and refused. The
CSV is untouched, since nothing was written to it. A CSV goes into a database through a loader, as
the worked examples did, and never through `connect`:


In [22]:
wrong.close()

conn = sqlite3.connect(DATABASE)
print("rows in the database:", conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0])
conn.close()


rows in the database: 35136


### sqlite3.OperationalError: no such table: readings


In [23]:
conn = sqlite3.connect(SCRATCH / "stations.bd")

conn.execute("SELECT COUNT(*) FROM readings")


OperationalError: no such table: readings

The name says `stations.bd`, and no file had that name, so `connect` did what it always does with a
missing file: it made a new, empty database, and the query found no table in it. The typo left that
empty file behind. A database that should already exist is safer opened by a URI with `mode=rw`,
which reads and writes but never creates, so a misspelled name fails at once instead of producing an
empty database that fails later:


In [24]:
conn.close()
typo = SCRATCH / "stations.bd"
print(typo.name, "exists:", typo.exists(), "| size:", typo.stat().st_size, "bytes")
typo.unlink()

try:
    sqlite3.connect(f"file:{typo}?mode=rw", uri=True)
except sqlite3.OperationalError as error:
    print("mode=rw, misspelled:", error)

conn = sqlite3.connect(f"file:{DATABASE}?mode=rw", uri=True)
print("mode=rw, spelled right:", conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0], "rows")
conn.close()


stations.bd exists: True | size: 0 bytes
mode=rw, misspelled: unable to open database file
mode=rw, spelled right: 35136 rows


### No error, and a mean that counts missing readings as zero: empty cells loaded as text


In [25]:
careless = sqlite3.connect(":memory:")            # a database held in memory, gone when it is closed
careless.execute("CREATE TABLE readings (station TEXT NOT NULL, hour TEXT NOT NULL, celsius REAL)")
with open(READINGS, newline="", encoding="utf-8") as file:
    careless.executemany("INSERT INTO readings VALUES (?, ?, ?)",
                         ((row["station"], row["hour"], row["celsius"]) for row in csv.DictReader(file)))

march = "SELECT AVG(celsius), MAX(celsius), COUNT(celsius) FROM readings WHERE station = ? AND hour LIKE ?"
mean, warmest, readings = careless.execute(march, ("Svalbard", "2025-03%")).fetchone()
print(f"Svalbard in March: mean {mean:.2f}, warmest {warmest!r}, readings {readings}")


Svalbard in March: mean -8.98, warmest '', readings 744


Nothing failed, and all three numbers are wrong. The `REAL` column turned every reading that looked
like a number into a number, but an empty string does not look like one, so the 24 empty readings
stayed text. `AVG` counts text that is not a number as 0, so the silent day pulled March's mean
toward zero degrees, `MAX` rated the empty string above every number, and `COUNT(celsius)` counted
empty strings as readings. Convert on the way in, as `reading` does, so that a missing value is
`NULL`, which all three skip:


In [26]:
careless.close()

conn = sqlite3.connect(DATABASE)
mean, warmest, readings = conn.execute(march, ("Svalbard", "2025-03%")).fetchone()
print(f"Svalbard in March: mean {mean:.2f}, warmest {warmest!r}, readings {readings}")
conn.close()


Svalbard in March: mean -9.28, warmest -4.0, readings 720


Last, the notebook is finished with its files, so this cell removes the scratch folder, with the CSV
files and every database in it:


In [27]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- `sqlite3.connect(path)` opens a database that is one file, creating the file when it does not
  exist, with nothing to install and no server to run.
- A CSV loaded into a table once, with its numbers converted and its empty cells stored as `NULL`,
  answers every later question without being read again.
- `SELECT` with `WHERE`, `GROUP BY`, `ORDER BY` and `LIMIT`, and the aggregates `MIN`, `MAX`, `AVG`
  and `COUNT`, say what you want, and SQLite works out how to get it.
- Values go into a statement through `?` placeholders, and new rows last only once they are
  committed.
- A new connection, or a copy of the file made while nothing writes to it, opens the same data at
  once.
- A database that should already exist is safer opened with `mode=rw`, so that a misspelled name is
  an error instead of a new, empty file.


## What is next

The **Connections and Cursors** notebook opens up the two objects every statement here went through:
the connection that holds the file open, and the cursor that runs a statement and hands back its
rows.


---

[sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Connections and Cursors](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/02-connections-and-cursors.ipynb) &#8594;
